# 3 — Read: t-digests → HHDC tensors, stage-timed

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/03_read_tensors.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/03_read_tensors.ipynb)
**in its default synthetic mode**: it writes a small t-digest store to a temp
directory with zagg's own write path, then reads it back — no credentials, no
cloud data. `SOURCE = "public"` (section 0) points the identical read cells at a
published store instead._

The third of three narrative notebooks
([#328](https://github.com/englacial/zagg/issues/328)). Notebook 1 built the
shard map, notebook 2 dispatched the fleet that wrote the product; this one
reads it back and casts it to the HHDC-style tensor blocks
[#265](https://github.com/englacial/zagg/issues/265) is about.

The reader contract (shipped by PR #339) is
`(tensor, mask, (offset, gain), morton_id)` per block:

| element | what it is |
| --- | --- |
| `tensor` | `(64, 64, n_bins)` per-bin counts — each cell's t-digest rasterized into a shared vertical window |
| `mask` | `(64, 64)` uint8 occupancy: `0` unobserved · `1` observed, no digest · `2` observed with data |
| `(offset, gain)` | the block's shared z-window: bin `i` covers `[offset + i·gain, offset + (i+1)·gain)` |
| `morton_id` | the block's coverage-cell morton id |

Cells are placed at the **bit deinterleave** of their nested rank
(`rank_to_rowcol`, mortie spec §8), not a row-major reshape — nested order is a
Z-order curve, so the naive reshape would scramble the block spatially. That fix
is what makes the tensor a *raster*.

## Why the stage split is the point

This is the unoptimized leg. Timing the read as one number tells you nothing;
splitting it into **fetch / decode / vertical-rasterize** tells you whether S3
or numpy is the wall, and therefore which of the open follow-ups
(concurrent leaf fetch vs. vectorized vertical binning) is worth doing first.

So sections 3–5 do the three stages **by hand**, with public API only, and
section 6 checks the hand-assembled tensor against `read_tensors` — the shipping
one-call path — for exact equality. The split is only worth reading if it
reproduces the real reader, so that check is not decoration.

## 0. Source switch

`SOURCE = "synthetic"` (default) builds a store on local disk with zagg's own
write path (`build_tdigest` + `write_ragged_to_zarr`) — the same layout a fleet
run produces, so the read code below is unchanged between the two modes. It is
a **local** store on purpose: a `MemoryStore` would make the fetch stage
meaningless.

`SOURCE = "public"` reads a published hive product anonymously. A hive store is
read **one leaf at a time** — a leaf zarr is exactly this layout scoped to one
shard — so the switch resolves the leaf path for the pinned shard and hands the
same `field` to the same calls.

> **Status (unverified here).** The public path is written against
> `s3://sliderule-public/zagg-bench/tdigest_healpix_o9_hive.zarr` — the store
> the [#265 thread](https://github.com/englacial/zagg/issues/265#issuecomment-5123695622)
> decoded — but **anonymous access to that bucket could not be verified from
> the session that wrote this notebook** (an unsigned client got
> `AccessDenied` on both LIST and GET). The Binder badge above therefore
> covers the **synthetic** default only, which is genuinely anonymous. See
> "Questions for review" on the PR.

In [ ]:
from pathlib import Path

import numpy as np

from zagg.notebook import StageTimer

SOURCE = "synthetic"  # or "public"

# Vertical rasterization knobs. 128 bins x 0.5 m = a 64 m window — the HHDC
# geometry (Ramirez-Jaime et al. 2024). The tail-trim quantiles are the
# parameter worth arguing about: the reference implementation ships 5/95 while
# the papers say 2/98, so they are named here rather than left implicit.
N_BINS, RESOLUTION = 128, 0.5
BOTTOM, TOP = 0.05, 0.95

# Block geometry, shared by both modes: one read chunk is a SIDE x SIDE block of
# cells at nesting DEPTH (SIDE == 2**DEPTH), and the cells are order CHILD_ORDER
# (~12.4 m) on both the synthetic grid and the published product. The synthetic
# grid's parent order lives with the scene in section 1 -- the published store's
# is 9, so it is not a constant the two modes can share.
SIDE, DEPTH, CHILD_ORDER = 64, 6, 19

# The published product (hive layout) and its pinned densest NEON shard —
# tests/data/benchmark/targets.json.
PUBLIC_STORE = "s3://sliderule-public/zagg-bench/tdigest_healpix_o9_hive.zarr"
PUBLIC_SHARD_KEY = 5347395636851376137

timer = StageTimer(f"read ({SOURCE})")
print(f"source: {SOURCE}")
print(f"window: {N_BINS} bins x {RESOLUTION} m = {N_BINS * RESOLUTION:g} m, "
      f"trimmed to the {BOTTOM:.0%}-{TOP:.0%} quantiles")

## 1. The scene — synthetic mode only

Sections 1 and 2 build the store; they no-op under `SOURCE = "public"`, which
opens a published one in section 2b instead. The read stages from section 3 on
are shared.

A single 64×64 block of order-19 HEALPix cells (~12.4 m each, so the block is
~800 m across) anchored at the **real** NEON SERC AOP coordinates — the same
place the benchmark AOI covers, so the shard key below is a genuine one from
the production grid, derived with `grid.assign` / `grid.shard_of` rather than
made up.

Each cell gets a photon-height sample drawn the way an airborne/spaceborne
lidar column looks: a tight ground return at the terrain height plus, where
there is canopy, a broad above-ground return skewed toward the crown. Two
canopy blobs sit on a gently tilted, slightly corrugated ground.

That gives section 7 something with a known answer: the bare-earth percentile
surface should recover the terrain and the canopy-height model should recover
the blobs.

**This is synthetic.** It is here so the notebook runs anonymously and so the
recovered surfaces can be scored against ground truth — not as a data product.

In [ ]:
PHOTONS_PER_CELL = 160
rng = np.random.default_rng(7)


def photon_column(ground_z, canopy_h):
    """One cell's photon heights: a ground return plus a skewed canopy return."""
    if canopy_h <= 0.0:
        return ground_z + rng.normal(0.0, 0.35, PHOTONS_PER_CELL)
    n_canopy = int(PHOTONS_PER_CELL * 0.55)
    return np.concatenate([
        ground_z + rng.normal(0.0, 0.35, PHOTONS_PER_CELL - n_canopy),
        ground_z + canopy_h * rng.beta(4.0, 1.6, n_canopy),
    ])


if SOURCE == "synthetic":
    PARENT_ORDER = 13  # synthetic grid only; the published product's is 9
    SERC_LAT, SERC_LON = 38.890, -76.560

    rows, cols = np.meshgrid(np.arange(SIDE), np.arange(SIDE), indexing="ij")
    terrain = 8.0 + 0.09 * cols + 0.05 * rows + 1.5 * np.sin(cols / 7.0)
    canopy = 22.0 * np.exp(-(((rows - 20) / 12.0) ** 2 + ((cols - 24) / 14.0) ** 2))
    canopy += 15.0 * np.exp(-(((rows - 46) / 9.0) ** 2 + ((cols - 44) / 11.0) ** 2))
    canopy[canopy < 2.0] = 0.0  # open ground outside the two stands

    # Not every cell is observed. A stylized gap -- a pond that returns nothing
    # and an across-track band with no coverage -- so the occupancy mask has
    # something to say. A real ICESat-2-only block is far sparser than this (a
    # profiling instrument leaves most columns unobserved, which is exactly why
    # the mask is a first-class output channel); this keeps the surfaces legible
    # while still exercising the {unobserved, observed} distinction.
    observed = np.ones((SIDE, SIDE), dtype=bool)
    observed[((rows - 34) / 7.0) ** 2 + ((cols - 12) / 9.0) ** 2 < 1.0] = False
    observed[(rows - cols > 44) | (cols - rows > 46)] = False

    print(f"{SIDE}x{SIDE} cells, {int(observed.sum())} observed "
          f"({observed.mean():.0%}), {PHOTONS_PER_CELL} photons each "
          f"= {int(observed.sum()) * PHOTONS_PER_CELL:,} synthetic photons")
    print(f"terrain {terrain.min():.1f}-{terrain.max():.1f} m, "
          f"canopy 0-{canopy.max():.1f} m above ground")

## 2. Write it with zagg's own write path — synthetic mode only

`build_tdigest` sketches each cell's photons; `write_ragged_to_zarr` writes the
block into the ragged variable-length-bytes field
([the #209 layout](https://github.com/englacial/zagg/blob/main/docs/ragged_layout.md)):
one vlen array on the cell grid, each populated cell holding the raw
little-endian bytes of its `(k, 2)` `(mean, weight)` digest, with the element
interpretation declared in the array's `ragged` attrs.

Cell order inside the block is **nested rank**, so the scene's `(row, col)`
grid is converted with `rowcol_to_rank` — the exact inverse of the
`rank_to_rowcol` the reader applies in section 5. Writing through the inverse
is what makes the round trip a real test of the deinterleave rather than a
tautology.

`delta=256` is the production centroid budget; at 160 photons per cell nothing
merges, so these digests are lossless (every centroid weight is 1) — the regime
[the #265 audit](https://github.com/englacial/zagg/issues/265#issuecomment-5123695622)
found 99.56% of live cells sitting in.

In [ ]:
import tempfile

import zarr
from zarr.storage import LocalStore

from zagg.config import PipelineConfig
from zagg.grids import HealpixGrid
from zagg.processing import write_ragged_to_zarr
from zagg.readers import rowcol_to_rank
from zagg.stats.tdigest import build_tdigest

if SOURCE == "synthetic":
    config = PipelineConfig(
        data_source={"groups": ["g"]},
        aggregation={
            "coordinates": {"morton": {"dtype": "uint64", "fill_value": 0}},
            "variables": {
                "h_tdigest": {
                    "function": "zagg.stats.tdigest.build_tdigest",
                    "source": "h",
                    "kind": "ragged",
                    "inner_shape": [2],  # each centroid is a (mean, weight) pair
                    "dtype": "float32",
                    "fill_value": 0,
                }
            },
        },
        output={"grid": {"type": "healpix",
                         "parent_order": PARENT_ORDER, "child_order": CHILD_ORDER}},
    )
    grid = HealpixGrid(PARENT_ORDER, CHILD_ORDER, layout="fullsphere", config=config)
    field = f"{grid.group_path}/h_tdigest"

    # A real shard key: the order-13 parent of the SERC point on the o19 grid.
    shard_key = int(grid.shard_of(grid.assign(np.array([SERC_LAT]), np.array([SERC_LON]))))
    print(f"shard {shard_key} (morton {grid.shard_label(shard_key)}), "
          f"{grid.cells_per_chunk} cells")

    by_rank = {
        rowcol_to_rank(r, c, DEPTH): photon_column(terrain[r, c], canopy[r, c])
        for r in range(SIDE)
        for c in range(SIDE)
        if observed[r, c]
    }
    ranks = sorted(by_rank)
    digests = [build_tdigest(np.asarray(by_rank[k]), delta=256) for k in ranks]

    tmp = Path(tempfile.mkdtemp(prefix="zagg-read-"))
    store = LocalStore(str(tmp / "scene.zarr"))
    grid.emit_template(store)

    block = grid.block_index(shard_key)
    cell_base = block[0] * grid.cells_per_chunk
    morton_arr = zarr.open_array(store, path=f"{grid.group_path}/morton", mode="r+")
    morton_arr[cell_base : cell_base + grid.cells_per_chunk] = grid.children(shard_key)
    write_ragged_to_zarr({"h_tdigest": (digests, ranks)}, store, grid=grid, chunk_idx=block)

    print(f"wrote {len(digests)} cell digests -> {tmp / 'scene.zarr'}")
    print(f"merge-free (lossless) cells: "
          f"{sum(1 for d in digests if (d[:, 1] == 1).all())}/{len(digests)}")

## 2b. Open a published product — public mode only

`SOURCE = "public"` skips sections 1–2 entirely and opens a published hive
product instead. The two things that differ from the synthetic store are the
store object and *which cells to read*: a leaf covers a whole shard, so the
populated block has to be found rather than assumed. Everything from section 3
on is identical.

In [ ]:
if SOURCE == "public":
    # A hive product is read one LEAF at a time: a leaf zarr is this same
    # layout scoped to one shard, so only the store argument changes.
    from zagg.hive import shard_leaf_path
    from zagg.readers import cell_index, read_tensors
    from zagg.store import open_store

    leaf = shard_leaf_path(PUBLIC_STORE, PUBLIC_SHARD_KEY)
    print(f"leaf: {leaf}")
    # skip_signature=True is the anonymous read (no AWS credentials).
    store = open_store(leaf, read_only=True, skip_signature=True)
    field = f"{CHILD_ORDER}/h_tdigest"

    # A leaf's cells axis is its WHOLE shard, not one read chunk: at the
    # published geometry (parent_order 9, child_order 19, chunk_inner 13) that
    # is 4**10 cells across 256 inner chunks of 64x64, most of them empty on a
    # sparse ICESat-2 shard. So find a POPULATED one rather than assuming cell
    # 0: read_tensors visits only the stored chunks, and cell_index turns the
    # chunk id it reports back into the global cells-axis offset (rank 0 is
    # (row, col) = (0, 0)) by searching the same stored spans.
    #
    # This is a separate pass on purpose. It re-reads the block, so it is NOT
    # part of the fetch/decode/rasterize split below -- selecting the block is
    # setup, not a read stage.
    _t, _m, _s, block_morton = next(read_tensors(store, field))
    cell_base = cell_index(store, field, block_morton, 0, 0)
    print(f"first populated block: morton {block_morton}, cells axis offset {cell_base}")

## 3. Stage — fetch

The array is opened and the block's cells are pulled in one slice. This is the
**I/O** term: on a sharded store it is the shard index suffix plus the inner
chunk objects the slice covers; the zstd decompression and the vlen framing
decode ride here too, since they are what the codec pipeline does on the way
out.

The element interpretation is read from the array's attrs, not hardcoded — the
layout is self-describing, which is the whole point of the `ragged` attrs block.

In [ ]:
from zagg.grids.base import RAGGED_ELEMENT_ATTR

arr = zarr.open_array(store, path=field, mode="r")
element = arr.attrs[RAGGED_ELEMENT_ATTR]["element"]
elem_dtype, elem_shape = np.dtype(element["dtype"]), tuple(element["shape"])
n_cells = SIDE * SIDE
print(f"{field}: {arr.shape[0]:,} cells, element {element}")

with timer.stage("fetch"):
    raw_cells = np.asarray(arr[cell_base : cell_base + n_cells])

populated = sum(1 for b in raw_cells if len(b))
stored_bytes = sum(len(b) for b in raw_cells)
print(f"{populated}/{n_cells} populated cells, {stored_bytes / 1024:.1f} KB of digest bytes")

## 4. Stage — decode

Each populated cell's bytes are a flat little-endian `(k, 2)` float32 array of
`(mean, weight)` centroids — the digest payload
([spec §2](https://github.com/englacial/zagg/blob/main/docs/specification.md)).
Decoding is a `frombuffer` + `reshape` per cell: no copy, no parsing. It is
timed separately precisely to show that it is *not* where the time goes.

In [ ]:
with timer.stage("decode"):
    cells = [
        (rank, np.frombuffer(raw, dtype=elem_dtype).reshape(elem_shape))
        for rank, raw in enumerate(raw_cells)
        if len(raw)
    ]

n_centroids = sum(len(d) for _rank, d in cells)
print(f"{len(cells)} digests, {n_centroids:,} centroids total "
      f"({n_centroids / max(len(cells), 1):.0f} per cell)")
print("first cell's centroids (mean, weight):")
print(cells[0][1][:4])

## 5. Stage — vertical rasterize

The expensive one, and the reason the split exists.

1. `chunk_z_range` trims every cell to its `BOTTOM`/`TOP` quantiles and anchors
   a fixed `N_BINS × RESOLUTION` window at the floor of the block-wide trimmed
   range. That window — `(offset, gain)` — is **shared by every cell in the
   block**, which is what makes the tensor stackable.
2. `rasterize_cell` evaluates the digest CDF at the bin edges and differences
   it, so each cell's `n_bins` vector is a histogram reconstructed from the
   sketch.
3. `rank_to_rowcol` places the cell at the bit deinterleave of its nested rank.

Step (2) runs a Python-level loop over cells, one `cdf_from_tdigest` call each —
the loop shape a vectorized read-side binning would remove (an open follow-up on
[#265](https://github.com/englacial/zagg/issues/265)). Whether that is worth
doing is exactly what this stage's share of the total answers.

In [ ]:
from zagg.readers import chunk_z_range, rank_to_rowcol, rasterize_cell

with timer.stage("vertical rasterize"):
    offset, n_bins, gain = chunk_z_range(
        [digest for _rank, digest in cells],
        n_bins=N_BINS,
        resolution=RESOLUTION,
        bottom=BOTTOM,
        top=TOP,
        fit="raise",
    )
    tensor = np.zeros((SIDE, SIDE, n_bins), dtype=np.uint32)
    mask = np.zeros((SIDE, SIDE), dtype=np.uint8)
    for rank, digest in cells:
        counts = np.rint(rasterize_cell(digest, offset, gain, n_bins))
        row, col = rank_to_rowcol(rank, DEPTH)
        tensor[row, col, :] = counts.astype(np.uint32)
        mask[row, col] = 2

print(f"tensor {tensor.shape} {tensor.dtype}, {int(tensor.sum()):,} counts")
print(f"z window: [{offset:g}, {offset + n_bins * gain:g}) m at {gain:g} m bins")

## 6. Does the hand split reproduce the real reader?

`read_tensors` does all three stages in one generator. If the hand-assembled
tensor and mask are not **exactly** equal to what it yields, the timings above
are measuring something else.

`has_exact_occupancy` is the mask discriminator: with the `coverage.moc`
occupancy sidecar a hive leaf carries, `0` genuinely means *unobserved* and `1`
means *observed but no stored digest*. Without it (any flat store, including the
synthetic one here) the mask degrades to `{0, 2}` and `0` means only "no digest
here" — the two regimes are indistinguishable from the array alone, which is why
a consumer keying on `mask == 1` has to ask first.

In [ ]:
from zagg.readers import has_exact_occupancy, read_tensors

ref_tensor, ref_mask, (ref_offset, ref_gain), morton_id = next(
    read_tensors(store, field, n_bins=N_BINS, resolution=RESOLUTION,
                 bottom=BOTTOM, top=TOP)
)

print(f"tensor identical: {np.array_equal(ref_tensor, tensor)}")
print(f"mask identical:   {np.array_equal(ref_mask, mask)}")
print(f"(offset, gain):   {(ref_offset, ref_gain)} vs {(offset, gain)}")
print(f"block morton id:  {morton_id}")
print(f"exact occupancy:  {has_exact_occupancy(store)}  "
      f"(False -> the mask is the degraded 2-state {{0, 2}} form)")
print(f"mask states present: {sorted(np.unique(mask).tolist())}")

## 7. Stage timings

The number this notebook exists to produce.

Read it as a ratio, not a wall clock: **fetch >> rasterize** says the transport
is the bottleneck and the lever is concurrent leaf fetch; **rasterize >> fetch**
says the per-cell CDF loop is, and the lever is vectorizing the read-side
vertical binning. Against a local store fetch is nearly free, so the synthetic
run lands firmly in the second regime — which is the honest caveat on the
synthetic mode: it tells you about the numpy term and almost nothing about the
S3 one.

In [ ]:
print(timer.summary())

In [ ]:
timer.as_dict()

## 8. Percentile surfaces: bare earth and canopy

A vertical histogram per cell is a *distribution*, so a surface is just a
quantile of it. Walk the cumulative counts up each column until the target
fraction is crossed and take that bin's height:

- **bare earth** — a low percentile (2%), the first returns off the ground,
- **canopy top** — a high percentile (98%),
- **CHM** (canopy height model) — the difference.

2/98 is what the HHDC papers use; the reference implementation ships 5/95. They
are a parameter here (`P_GROUND` / `P_CANOPY`), not a constant, because the
choice visibly moves the answer on sparse columns.

Note this works on the *tensor* — binned counts — so it inherits the bin
resolution as its floor (0.5 m here). `zagg.stats.tdigest.quantile_from_tdigest`
on the digests is the unbinned, more exact route; the cell below scores both
against the known truth so the cost of binning is visible rather than assumed.

In [ ]:
def percentile_surface(tensor, offset, gain, p):
    """Height at which the cumulative count of each cell's column crosses ``p``."""
    counts = tensor.astype(np.float64)
    total = counts.sum(axis=2)
    crossed = (np.cumsum(counts, axis=2) < (p * total)[..., None]).sum(axis=2)
    z = offset + (crossed + 0.5) * gain
    return np.where(total > 0, z, np.nan)


P_GROUND, P_CANOPY = 0.02, 0.98
bare_earth = percentile_surface(tensor, offset, gain, P_GROUND)
canopy_top = percentile_surface(tensor, offset, gain, P_CANOPY)
chm = canopy_top - bare_earth

if SOURCE == "synthetic":
    from zagg.stats.tdigest import quantile_from_tdigest

    # The unbinned route, for comparison: quantiles straight off the digests.
    exact_ground = np.full((SIDE, SIDE), np.nan)
    for rank, digest in cells:
        row, col = rank_to_rowcol(rank, DEPTH)
        exact_ground[row, col] = quantile_from_tdigest(digest, P_GROUND)

    print(f"bare earth vs known terrain — binned MAE  {np.nanmean(np.abs(bare_earth - terrain)):.3f} m")
    print(f"bare earth vs known terrain — digest MAE  {np.nanmean(np.abs(exact_ground - terrain)):.3f} m")
    print(f"CHM vs known canopy        — binned MAE  {np.nanmean(np.abs(chm - canopy)):.3f} m")
    print(f"(bin resolution is {gain:g} m, so ~{gain / 2:g} m is the binned floor)")

print(f"\nCHM range: {np.nanmin(chm):.1f} - {np.nanmax(chm):.1f} m")

In [ ]:
import matplotlib.pyplot as plt

panels = [
    (bare_earth, f"bare earth (p{P_GROUND:.0%})", "terrain", "m"),
    (canopy_top, f"canopy top (p{P_CANOPY:.0%})", "terrain", "m"),
    (chm, "canopy height model", "viridis", "m"),
    (mask, "occupancy mask", "gray", "state"),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), constrained_layout=True)
for ax, (surface, title, cmap, label) in zip(axes, panels):
    im = ax.imshow(surface, origin="lower", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("col (deinterleaved)")
    fig.colorbar(im, ax=ax, label=label, shrink=0.85)
axes[0].set_ylabel("row (deinterleaved)")
fig.suptitle(f"percentile surfaces from the {SIDE}x{SIDE}x{n_bins} block "
             f"(morton {morton_id})")
plt.show()

In [ ]:
# One cell's reconstructed vertical profile: the ground spike plus the canopy
# shoulder that the two percentiles bracket. Pick the block's densest column so
# the cell is populated whatever the source is (in synthetic mode that lands in
# one of the two stands).
row, col = np.unravel_index(int(np.argmax(tensor.sum(axis=2))), (SIDE, SIDE))
edges = offset + gain * np.arange(n_bins)
fig, ax = plt.subplots(figsize=(5, 4.2), constrained_layout=True)
ax.step(tensor[row, col], edges, where="mid")
ax.axhline(bare_earth[row, col], color="tab:brown", ls="--",
           label=f"bare earth p{P_GROUND:.0%}")
ax.axhline(canopy_top[row, col], color="tab:green", ls="--",
           label=f"canopy top p{P_CANOPY:.0%}")
ax.set_xlabel("photons per bin")
ax.set_ylabel("height (m)")
ax.set_title(f"cell (row {row}, col {col})")
ax.legend()
plt.show()

## Where this goes

- The `mask` channel is a first-class output, not a nicety: ICESat-2 is a
  profiling instrument, so an IS2-only block is mostly *unobserved columns*.
  Distinguishing "unobserved" from "observed, zero returns" is the thing the
  reference `.npz` HHDC format cannot do (it zero-fills both), and it is what
  makes these blocks usable as compressive-sampling inputs.
- The two open read-side follow-ups on
  [#265](https://github.com/englacial/zagg/issues/265) are exactly the two terms
  section 7 splits: vectorizing the per-cell vertical binning, and concurrent
  leaf fetch.
- `notebooks/tdigest_reader_example.ipynb` covers the rest of the reader API —
  the `fit` policy for blocks whose trimmed range overflows the window,
  `block_order=` for assembling 128×128 blocks from whole chunks, `read_cell`
  random access, and `read_raw_values` lossless recovery.